In [ ]:
import json
import os
import csv
from datetime import datetime,timezone
from pathlib import Path
import anthropic
from anthropic import Anthropic
import claude_key
from dotenv import load_dotenv

anthro_key = claude_key.config_data["key"]
client = anthropic.Anthropic(api_key=anthro_key)


# ---------- Config ----------
STIMULI_PATH = Path("stimuli/stimuli.json")
PROMPTS_PATH = Path("prompts/prompts.json")
OUTPUT_CSV = Path("responses/responses_raw.csv")

MODEL_NAME = "claude-sonnet-4-5"
PROMPT_TYPES_TO_RUN = ["open", "guided"]   # must match prompt_type values in prompts.json
N_RUNS_PER_CONDITION = 3                   # e.g., 3-10 later
TEMPERATURE = 0.7
MAX_TOKENS = 300

# ---------- Helpers ----------
def load_stimuli(path: Path) -> list[dict]:
    with path.open("r", encoding="utf-8") as f:
        data = json.load(f)
    if not isinstance(data, list):
        raise ValueError("stimuli.json must be a JSON array (a list of stimulus objects).")
    return data

def load_prompt_templates(path: Path) -> dict[str, str]:
    """Returns mapping: prompt_type -> template string"""
    with path.open("r", encoding="utf-8") as f:
        data = json.load(f)

    prompts = data.get("prompts")
    if not isinstance(prompts, list):
        raise ValueError("prompts.json must contain a top-level key 'prompts' that is a list.")

    templates = {}
    for p in prompts:
        ptype = p.get("prompt_type")
        template = p.get("template")
        if isinstance(ptype, str) and isinstance(template, str):
            templates[ptype] = template

    return templates

def render_template(template: str, stimulus: dict) -> str:
    """
    Replaces placeholders like {sentence}, {target_word}, etc.
    If a placeholder isn't present in stimulus, it will KeyError .
    """
    return template.format(**stimulus)

def ensure_csv_has_header(path: Path, fieldnames: list[str]) -> None:
    if path.exists():
        return
    with path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()

def append_row(path: Path, fieldnames: list[str], row: dict) -> None:
    with path.open("a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writerow(row)

# ---------- Main ----------
def main():
    
    stimuli = load_stimuli(STIMULI_PATH)
    templates = load_prompt_templates(PROMPTS_PATH)

    # Verify requested prompt types exist
    for ptype in PROMPT_TYPES_TO_RUN:
        if ptype not in templates:
            raise ValueError(f"prompt_type '{ptype}' not found in prompts.json. Found: {list(templates.keys())}")



    # Decide what columns you want to log (add more anytime)
    fieldnames = [
        "response_id",
        "timestamp_utc",
        "model",
        "temperature",
        "max_output_tokens",
        "prompt_type",
        "run_id",
        "prompt_id",
        "target_word",
        "sense_id",
        "sense_family",
        "context_type",
        "sentence",
        "prompt_text",
        "response_text",
    ]
    ensure_csv_has_header(OUTPUT_CSV, fieldnames)

    total = len(stimuli) * len(PROMPT_TYPES_TO_RUN) * N_RUNS_PER_CONDITION
    i = 0

    for stimulus in stimuli:
        for prompt_type in PROMPT_TYPES_TO_RUN:
            template = templates[prompt_type]
            prompt_text = render_template(template, stimulus)

            for run_id in range(1, N_RUNS_PER_CONDITION + 1):
                i += 1
                print(f"[{i}/{total}] {MODEL_NAME} | {stimulus['prompt_id']} | {prompt_type} | run {run_id}")

                resp = client.messages.create(
                    model=MODEL_NAME,
                    temperature=TEMPERATURE,
                    max_tokens=MAX_TOKENS,
                    messages=[{"role": "user", "content": prompt_text}],
                )

                # output_text is a convenience; it returns the text content
                response_text = "".join(
                    block.text for block in resp.content if getattr(block, "type", None) 
                    == "text")

                row = {
                    "response_id": getattr(resp, "id", None),
                    "timestamp_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
                    "model": MODEL_NAME,
                    "temperature": TEMPERATURE,
                    "max_output_tokens": MAX_TOKENS,
                    "prompt_type": prompt_type,
                    "run_id": run_id,
                    "prompt_id": stimulus.get("prompt_id"),
                    "target_word": stimulus.get("target_word"),
                    "sense_id": stimulus.get("sense_id"),
                    "sense_family": stimulus.get("sense_family"),
                    "context_type": stimulus.get("context_type"),
                    "sentence": stimulus.get("sentence"),
                    "prompt_text": prompt_text,
                    "response_text": response_text,
                }

                append_row(OUTPUT_CSV, fieldnames, row)

    print(f"\nDone. Saved to: {OUTPUT_CSV.resolve()}")

if __name__ == "__main__":
    main()


[1/276] claude-sonnet-4-5 | bank_river_01 | open | run 1


C:\Users\matth\AppData\Local\Temp\ipykernel_704\539375528.py:129: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp_utc": datetime.utcnow().isoformat(timespec="seconds") + "Z",


[2/276] claude-sonnet-4-5 | bank_river_01 | open | run 2
[3/276] claude-sonnet-4-5 | bank_river_01 | open | run 3
[4/276] claude-sonnet-4-5 | bank_river_01 | guided | run 1
[5/276] claude-sonnet-4-5 | bank_river_01 | guided | run 2
[6/276] claude-sonnet-4-5 | bank_river_01 | guided | run 3
[7/276] claude-sonnet-4-5 | bank_river_02 | open | run 1
[8/276] claude-sonnet-4-5 | bank_river_02 | open | run 2
[9/276] claude-sonnet-4-5 | bank_river_02 | open | run 3
[10/276] claude-sonnet-4-5 | bank_river_02 | guided | run 1
[11/276] claude-sonnet-4-5 | bank_river_02 | guided | run 2
[12/276] claude-sonnet-4-5 | bank_river_02 | guided | run 3
[13/276] claude-sonnet-4-5 | bank_river_03 | open | run 1
[14/276] claude-sonnet-4-5 | bank_river_03 | open | run 2
[15/276] claude-sonnet-4-5 | bank_river_03 | open | run 3
[16/276] claude-sonnet-4-5 | bank_river_03 | guided | run 1
[17/276] claude-sonnet-4-5 | bank_river_03 | guided | run 2
[18/276] claude-sonnet-4-5 | bank_river_03 | guided | run 3
[19/